In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import pandas as pd
from bs4 import BeautifulSoup
import re
import requests
import datetime
from selenium import webdriver
from time import sleep
import os



In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'MS MFSC' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running MS MFSC Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder,
         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert
         }
chromeOptions.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


In [4]:
#------------------------------------------------ Begin_Variable ----------------------------------------


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

         regulatorName + ' 1': 'https://www.fscmontserrat.org/regulated-entities/international-offshore-banking/list-of-banks/',
         regulatorName + ' 2': 'https://www.fscmontserrat.org/regulated-entities/insurance-business/insurance-businesses/',
         regulatorName + ' 3': 'https://www.fscmontserrat.org/regulated-entities/how-to-apply-for-a-licence/26-2/list-of-banks/',
         regulatorName + ' 4': 'https://www.fscmontserrat.org/regulated-entities/company-management/',

        }

headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36"
    }
        

In [5]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict

In [ ]:
#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):
    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")

    resp = requests.get(regdict[reg],  headers=headers, timeout=30)
    resp.raise_for_status()
    print(resp.url)
    print(resp.status_code)
    soup = BeautifulSoup(resp.text, "html.parser")
    maincontent = soup.find('div', id='main')
    if maincontent:
        content_ = maincontent.find('div', id='content')
        if content_:
            if reg!= regulatorName + ' 4':
                title_ = content_.find(
                    'h1')
                #print(title_.text)
                content_lis = content_.find_all('li')
                if content_lis:
                    for li in content_lis:
                        name_ = li.get_text(strip=True)
                        sqldict['Name'].append(name_)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['Typology'].append(title_.text if title_ else '')
                        sqldict['RegCtry'].append(reg.split(' ')[0])
                        sqldict['RegCode'].append(reg.split(' ')[1])
                        sqldict['ListCode'].append('1')
                        sqldict['RegulationType'].append('Regulated')
                        sqldict['ListName'].append('Regulated entities')
                        sqldict = bourange_same_length_array(sqldict)
                        #print(name_)
                else:
                    name_ = content_.find('p').text
                    sqldict['Name'].append(name_)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['Typology'].append(title_.text if title_ else '')
                    sqldict['RegCtry'].append(reg.split(' ')[0])
                    sqldict['RegCode'].append(reg.split(' ')[1])
                    sqldict['ListCode'].append('1')
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['ListName'].append('Regulated entities')
                    sqldict = bourange_same_length_array(sqldict)
            
            else:
                title_ = content_.find(
                    'h1')
                print(title_.text)
                content_ps = content_.find_all('h4')
                if content_ps:
                    for p in content_ps:
                        name_ = p.get_text(strip=True)
                        details_ = p.find_next_sibling('p')
                        name_ = name_.split('.')[-1].strip()
                        # print(details_)
                        text = details_.get_text("\n", strip=True)

                        # address: all lines before "Tel:"
                        lines = text.split("\n")
                        tel_line = next((l for l in lines if l.lower().startswith("tel:")), "")
                        email_line = next((l for l in lines if l.lower().startswith("email:")), "")
                        web_line = next((l for l in lines if l.lower().startswith("web:")), "")
                        address_lines = []
                        for l in lines:
                            if l.lower().startswith("tel:"):
                                break
                            address_lines.append(l)
                        address_ = ", ".join(address_lines)

                        # phones: everything after "Tel:"
                        tel = tel_line.replace("Tel:", "").strip().split("|")[0] if tel_line else ""

                        # email: anchor text or after "Email:"
                        email_ = ""
                        a_email = details_.find("a", href=re.compile(r"^mailto:", re.I))
                        if a_email:
                            email_ = a_email.get_text(strip=True)
                        web_ = ""

                        a_web = details_.find("a", href=re.compile(r"^http", re.I))
                        if a_web:
                            web_ = a_web.get_text(strip=True)

                        #print({"address_": address_, "tel": tel, "email_": email_, "web_": web_})

                        sqldict['Name'].append(name_)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['Typology'].append(title_.text if title_ else '')
                        sqldict['RegCtry'].append(reg.split(' ')[0])
                        sqldict['RegCode'].append(reg.split(' ')[1])
                        sqldict['ListCode'].append('1')
                        sqldict['RegulationType'].append('Regulated')
                        sqldict['ListName'].append('Regulated entities')
                        sqldict['Address_1'].append(address_)
                        sqldict['Phone'].append(tel)
                        sqldict['Email'].append(email_)
                        sqldict['Website'].append(web_)
                        sqldict = bourange_same_length_array(sqldict)
                        

[INFO] : Working 1/4 _(MS MFSC 1)_ 
https://www.fscmontserrat.org/regulated-entities/international-offshore-banking/list-of-banks/
200
[INFO] : Working 2/4 _(MS MFSC 2)_ 
https://www.fscmontserrat.org/regulated-entities/insurance-business/insurance-businesses/
200
[INFO] : Working 3/4 _(MS MFSC 3)_ 
https://www.fscmontserrat.org/regulated-entities/how-to-apply-for-a-licence/26-2/list-of-banks/
200
[INFO] : Working 4/4 _(MS MFSC 4)_ 
https://www.fscmontserrat.org/regulated-entities/company-management/
200
Company Management


In [ ]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename, index=False)
driver.quit()
sleep(3)

In [ ]:

import requests
API_TOKEN = ""
URL = "https://Api.bvdinfo.com/v1/orbis/companies/match"



headers = {
    "ApiToken": API_TOKEN,
    "Content-Type": "application/json"
}
# df is your dataframe with a "Name" column
for i, row in df.iterrows():
    ITERATE_NAME = str(row["Name"]).strip() if row["Name"] is not None else ""
    fax_ = str(row["Fax"]).strip() if row["Fax"] is not None else ""
    tel_ = str(row["Phone"]).strip() if row["Phone"] is not None else ""
    address_ = str(row["Address_1"]).strip() if row["Address_1"] is not None else ""
    email_ = str(row["Email"]).strip() if row["Email"] is not None else ""
    website_ = str(row["Website"]).strip() if row["Website"] is not None else ""
    payload = {
        "MATCH": {
            "Criteria": {
                "Name": ITERATE_NAME,
                "Country": "MS",
                # "EMailOrWebsite": email_ or website_,
                # "Address": address_,
                # "PhoneOrFax": fax_ or tel_
            },
            "Options": {
                "ScoreLimit": 0.85
            }
        },
        "SELECT": [
            "Match.Hint",
            "Match.Score",
            "Match.Name",
            "Match.Name_Local",
            "Match.Address",
            "Match.Postcode",
            "Match.City",
            "Match.Country",
            "Match.Status",
            "Match.National_Id",
            "Match.NationalIdLabel",
            "Match.LegalForm",
            "Match.BvDId",
            "Match.PhoneOrFax",
            "Match.EmailOrWebsite"
        ]
    }

    response = requests.post(URL, headers=headers, json=payload, timeout=60)
    #print("Status:", response.status_code)

    try:
        data = response.json()
        bvd_id = ""
        #print(json.dumps(data, indent=2))
        if data and data[0].get("Hint", "").lower() != "unlikely":
            bvd_id = data[0].get("BvDId", "")
    except Exception:
        #print(response.text)
        bvd_id = ""
    #print(bvd_id)
    df.at[i, "bvdid"] = bvd_id if bvd_id else ""




GBIM131969C
GB04090062
GB*973035503



GBLEI1037364




GBFEB50446





In [23]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check,BvDId
0,,,,International Banks,,"Credit and Commerce Bank, Inc.",,,,,...,,,,,,,,,,
1,GBIM131969C,,,International Banks,,Capital International Bank Inc.,,,,,...,,,,,,,,,,
2,GB04090062,,,International Banks,,Global Bank Overseas Limited,,,,,...,,,,,,,,,,
3,GB*973035503,,,International Banks,,Lafise Bank Limited,,,,,...,,,,,,,,,,
4,,,,Insurance Companies,,British American Insurance Company Limited,,,,,...,,,,,,,,,,
5,,,,Insurance Companies,,Caribbean Alliance Insurance Company Ltd,,,,,...,,,,,,,,,,
6,,,,Insurance Companies,,Colonial Life Insurance Company (Trinidad) CLI...,,,,,...,,,,,,,,,,
7,GBLEI1037364,,,Insurance Companies,,Nagico Insurance Company Ltd,,,,,...,,,,,,,,,,
8,,,,Insurance Companies,,Guardian General Insurance Ltd,,,,,...,,,,,,,,,,
9,,,,Insurance Companies,,CG United Insurance Limited (formerly Massy Un...,,,,,...,,,,,,,,,,
